In [36]:
%load_ext autoreload
%autoreload 2
from src.data_utils import DataUtils
import yaml
import pandas as pd


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
# Загрузка конфига
with open("configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [38]:
from src.next_token_dataset import NextTokenDataset
from transformers import BertTokenizerFast
from torch.utils.data import DataLoader
import torch
from src.lstm_model import LstmModel

# Создание csv файлов
# DataUtils.samples_create(config['dataset'])

In [ ]:

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

df_train = pd.read_csv(config['dataset']['path'] + '/train.csv')
df_val = pd.read_csv(config['dataset']['path'] + '/val.csv')

train_dataset = NextTokenDataset(df_train, tokenizer, max_len=16)
val_dataset = NextTokenDataset(df_val, tokenizer, max_len=16)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

In [47]:
from src.eval_transformer_pipeline import eval_transformer_pipeline

model = LstmModel(vocab_size=tokenizer.vocab_size, hidden_dim=128)

eval_transformer_pipeline(
    config=config,
    model=model,
    tokenizer=tokenizer,
    train_loader=train_loader,
    val_loader=val_loader
)

Epoch 1/10 | Train Loss: nan | Val Loss: 10.8229 | Val Acc: 0.0000
Epoch 2/10 | Train Loss: nan | Val Loss: 10.8229 | Val Acc: 0.0000
Epoch 3/10 | Train Loss: nan | Val Loss: 10.8229 | Val Acc: 0.0000
Epoch 4/10 | Train Loss: nan | Val Loss: 10.8229 | Val Acc: 0.0000
Epoch 5/10 | Train Loss: nan | Val Loss: 10.8229 | Val Acc: 0.0000
Epoch 6/10 | Train Loss: nan | Val Loss: 10.8229 | Val Acc: 0.0000
Epoch 7/10 | Train Loss: nan | Val Loss: 10.8229 | Val Acc: 0.0000
Epoch 8/10 | Train Loss: nan | Val Loss: 10.8229 | Val Acc: 0.0000
Epoch 9/10 | Train Loss: nan | Val Loss: 10.8229 | Val Acc: 0.0000
Epoch 10/10 | Train Loss: nan | Val Loss: 10.8229 | Val Acc: 0.0000


In [42]:
model.eval()
for i in range(10):
    text = df_train.iloc[i]
    if isinstance(text, (pd.Series, dict)):
        text = text[0]  # если DataFrame с одной колонкой
    prompt = text.split()[: len(text.split()) * 3 // 4]
    prompt_str = " ".join(prompt)

    # Токенизируем "частичный" текст
    input_ids = tokenizer.encode(prompt_str, return_tensors="pt", truncation=True, max_length=32)

    # Генерируем дополнение
    generated_ids = model.generate(input_ids, max_new_tokens=10)
    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print(f"🟢 Original: {text}")
    print(f"🔹 Prompt: {prompt_str}")
    print(f"🔸 Generated: {generated_text}\n{'-'*80}")


🟢 Original: bartonbishop im doing my best theres no way for me to get a hold of my boss and i dont think shes checking email
🔹 Prompt: bartonbishop im doing my best theres no way for me to get a hold of my boss and
🔸 Generated: bartonbishop im doing my best theres no way for me to get a hold of my boss and sly smoking guaranteedither Trilogy strive�metic THERE trained
--------------------------------------------------------------------------------
🟢 Original: misspelled those wrong
🔹 Prompt: misspelled those
🔸 Generated: misspelled those active urgency ware nationalism caption OW crochetphas",duction
--------------------------------------------------------------------------------
🟢 Original: 88 thats my twitter value
🔹 Prompt: 88 thats my
🔸 Generated: 88 thats my partnerships registrationFG Princize collect stack holog shorts pork
--------------------------------------------------------------------------------
🟢 Original: sunsfan69 been in there a few times usually right after the gym


/var/folders/2p/1g0tbznd1zg9118y89qrclsc0000gn/T/ipykernel_7490/1940004646.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  text = text[0]  # если DataFrame с одной колонкой
